In [6]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Загрузка данных (относительный путь от папки notebooks к data)
df = pd.read_csv("../data/tips.csv")

# Добавим немного пропусков для демонстрации работы импьютера
np.random.seed(42)
df.loc[df.sample(frac=0.05).index, "total_bill"] = np.nan
df.loc[df.sample(frac=0.05).index, "sex"] = np.nan

df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [7]:
# Выделяем целевую переменную и признаки
X = df.drop(columns=["tip"])
y = df["tip"]

# 1. Группируем признаки по типам
num_features = ["total_bill", "size"]
cat_features = ["sex", "smoker", "day", "time"]

# 2. Разделяем выборку строго ДО вычисления любых статистик
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Размер обучающей выборки X_train: {X_train.shape}")
print(f"Размер тестовой выборки X_test:   {X_test.shape}")

Размер обучающей выборки X_train: (195, 6)
Размер тестовой выборки X_test:   (49, 6)


In [8]:
# Пайплайн для числовых признаков
num_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Пайплайн для категориальных признаков
cat_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="constant", fill_value="unknown"),
        ),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

# Объединение трансформеров
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_features),
        ("cat", cat_pipeline, cat_features),
    ]
)

In [9]:
# fit_transform вызывается ТОЛЬКО на train
X_train_processed = preprocessor.fit_transform(X_train)

# на test вызывается ТОЛЬКО transform
X_test_processed = preprocessor.transform(X_test)

print(f"Форма X_train после обработки: {X_train_processed.shape}")
print(f"Форма X_test после обработки:  {X_test_processed.shape}")

# Проверка на совпадение количества колонок
assert (
    X_train_processed.shape[1] == X_test_processed.shape[1]
), "Количество колонок не совпадает!"
print(
    "\n[OK] Пайплайн применен корректно, утечка данных отсутствует!"
)

Форма X_train после обработки: (195, 13)
Форма X_test после обработки:  (49, 13)

[OK] Пайплайн применен корректно, утечка данных отсутствует!


### Обоснование выбранных стратегий (Вариант B):

1. **Защита от утечки данных:** Разбиение `train_test_split` выполнено в самом начале. Метод `fit` обучался строго на `X_train`.
2. **Заполнение пропусков у числовых признаков:** Выбрана медиана (`strategy='median'`), так как она не чувствительна к крайним выбросам в отличие от среднего арифметического.
3. **Заполнение пропусков у категориальных признаков:** Пропущенные значения заполняются константой `'unknown'`, чтобы модель воспринимала факт отсутствия данных как отдельную категорию.
4. **Кодирование и обработка ошибок:** `OneHotEncoder(handle_unknown='ignore')` преобразует категории в бинарные векторы и предотвращает падение программы при появлении новых неизвестных категорий в тестовых данных.